# CAT Saathi — train the intent model

Run the cells top to bottom. **Runtime → Change runtime type → T4 GPU** first.

Checkpoints go to your Google Drive every few minutes. If Colab disconnects, reconnect and
**run all cells again** — training picks up from the last checkpoint, not from zero.


## 1. GPU check


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU - switch runtime to T4'


## 2. Connect Google Drive (this is where progress is saved)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/cat-saathi/intent-run'
print('checkpoints will be saved in', RUN_DIR)


## 3. Get the code


In [ ]:
!git clone -q https://github.com/krishagarwal314/sahayak-cat-operator-assistant.git /content/saathi 2>/dev/null || git -C /content/saathi pull -q
%cd /content/saathi/backend
!pip -q install 'numpy<2' tqdm > /dev/null && echo ready


## 4. Build the training data
Five focus intents get hundreds of English phrasings each; the rest are lighter.


In [ ]:
!python -m app.ai.intent.build_dataset


## 5. Train
One progress bar for the whole run. Takes roughly 5–10 minutes on a T4.
If it stops, just run this cell again — it resumes.


In [ ]:
!python -m app.ai.intent.train --out "$RUN_DIR" --epochs 6


## 6. Test it
Type anything — English, Hindi or Hinglish.


In [ ]:
import sys, json, torch
sys.path.insert(0, '/content/saathi/backend')
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from app.ai.intent.normalize import normalize
best = RUN_DIR + '/best'
tok = AutoTokenizer.from_pretrained(best)
model = AutoModelForSequenceClassification.from_pretrained(best).eval()
labels = {int(k): v for k, v in json.load(open(best + '/labels.json')).items()}

def ask(text):
    with torch.no_grad():
        probs = torch.softmax(model(**tok(normalize(text), return_tensors='pt')).logits[0], -1)
    top = probs.topk(2)
    print(f'{text:45s} -> {labels[top.indices[0].item()]:20s} {top.values[0]:.0%}   (2nd: {labels[top.indices[1].item()]} {top.values[1]:.0%})')

for q in ['how much fuel is left', 'is anything wrong with the machine', 'is it safe to work',
          'how long will this take', 'how do i start the machine', 'kitna diesel bacha hai',
          'मशीन में कोई खराबी है क्या', 'how long did i idle', 'how do i make tea']:
    ask(q)


## 7. Take the model to Lightning

**Option A — Hugging Face (easiest).** Paste your HF token (write access) below. It uploads privately.


In [ ]:
HF_TOKEN = ''          # paste a WRITE token from huggingface.co/settings/tokens
REPO = 'cat-saathi-intent'
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
repo_id = f"{api.whoami()['name']}/{REPO}"
api.create_repo(repo_id, private=True, exist_ok=True)
api.upload_folder(folder_path=RUN_DIR + '/best', repo_id=repo_id)
print('uploaded to', repo_id)
print('\nOn Lightning run:')
print(f"  cd backend && python -c \"from huggingface_hub import snapshot_download; snapshot_download('{repo_id}', local_dir='models/intent-classifier', token='YOUR_TOKEN')\"")


**Option B — download a zip** and upload it to Lightning, then unzip into `backend/models/intent-classifier`.


In [ ]:
!cd "$RUN_DIR" && rm -f intent-classifier.zip && cd best && zip -qr ../intent-classifier.zip . && ls -lh ../intent-classifier.zip
from google.colab import files
files.download(RUN_DIR + '/intent-classifier.zip')
